# LeetCode 3426 — Manhattan Distances of All Arrangements of Pieces bằng Python

- Link: https://leetcode.com/problems/manhattan-distances-of-all-arrangements-of-pieces/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp**, **đổi thứ tự lấy tổng** (đếm theo cặp ô), **tách tọa độ** của khoảng cách Manhattan, **tổ hợp modulo** (giai thừa + nghịch đảo Fermat)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def distanceSum(self, m: int, n: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Đổi thứ tự tổng: thay vì duyệt mọi cách xếp rồi mọi cặp quân, duyệt **mọi cặp ô** và đếm số cách xếp chứa cả hai ô đó: `C(mn − 2, k − 2)`.
2. Tách khoảng cách Manhattan thành phần **hàng** và **cột**; mỗi phần tính độc lập.
3. Tính `Σ_{cặp hàng} |r1 − r2| = Σ_{d=1}^{m−1} d·(m − d) = m(m² − 1)/6`.
4. Tính tổ hợp modulo `10^9 + 7` bằng giai thừa và **nghịch đảo modulo** (định lý Fermat nhỏ).
5. Tránh lỗi chia trong modulo (chia 6, chia giai thừa).

## 2. Đề bài & ví dụ

Lưới `m × n`, đặt `k` quân giống nhau (mỗi ô tối đa một quân). Tính **tổng khoảng cách Manhattan giữa mọi cặp quân, cộng trên mọi cách xếp hợp lệ**, modulo `10^9 + 7`.

**Ví dụ 1:** `m = 2, n = 2, k = 2` → `8`. **Ví dụ 2:** `m = 1, n = 4, k = 3` → `20`.

**Ràng buộc:** `1 ≤ m, n ≤ 10^5`, `2 ≤ mn ≤ 10^5`, `2 ≤ k ≤ mn`.

In [1]:
MOD = 10 ** 9 + 7
examples = [(2, 2, 2, 8), (1, 4, 3, 20)]
for m, n, k, exp in examples:
    print(m, n, k, '-> kỳ vọng', exp)

2 2 2 -> kỳ vọng 8
1 4 3 -> kỳ vọng 20


## 3. Kiến thức nền

### 3.1. Đổi thứ tự lấy tổng

`Σ_{cách xếp S} Σ_{cặp {p, q} ⊆ S} dist(p, q) = Σ_{cặp ô {p, q}} dist(p, q) · #{S : p, q ∈ S}`. Số cách xếp chứa cả `p` và `q` là chọn `k − 2` ô còn lại trong `mn − 2` ô: `C(mn − 2, k − 2)`.

### 3.2. Tổng khoảng cách trên một trục

In [2]:
def axis_sum_brute(m):
    return sum(abs(a - b) for a in range(m) for b in range(a + 1, m))

def axis_sum_loop(m):
    return sum(d * (m - d) for d in range(1, m))

def axis_sum_closed(m):
    return m * (m * m - 1) // 6

print([(m, axis_sum_brute(m), axis_sum_loop(m), axis_sum_closed(m)) for m in range(1, 7)])

[(1, 0, 0, 0), (2, 1, 1, 1), (3, 4, 4, 4), (4, 10, 10, 10), (5, 20, 20, 20), (6, 35, 35, 35)]


Có `m − d` cặp hàng cách nhau đúng `d`. Đẳng thức đóng: `Σ d(m − d) = m · m(m−1)/2 − (m−1)m(2m−1)/6 = m(m² − 1)/6`.

### 3.3. Tổ hợp modulo số nguyên tố

`C(N, r) = N! / (r! (N − r)!)`. Modulo số nguyên tố `p`, chia ⇔ nhân nghịch đảo: `a^{-1} ≡ a^{p−2} (mod p)`.

In [3]:
from math import comb

def comb_mod(N, r, mod=MOD):
    if r < 0 or r > N:
        return 0
    num = den = 1
    for i in range(r):
        num = num * (N - i) % mod
        den = den * (i + 1) % mod
    return num * pow(den, mod - 2, mod) % mod

print(comb_mod(10, 3), comb(10, 3), comb_mod(10 ** 5, 777) == comb(10 ** 5, 777) % MOD)

120 120 True


### 3.4. Chứng minh công thức và tính chia hết cho 6

`Σ_{d=1}^{m−1} d(m − d) = m · Σ d − Σ d² = m · (m−1)m/2 − (m−1)m(2m−1)/6 = (m−1)m · (3m − (2m − 1))/6 = (m−1)m(m+1)/6`.

Tích `(m−1)m(m+1)` của ba số nguyên liên tiếp luôn chia hết cho `2` và `3` ⇒ chia hết cho 6, nên kết quả là số nguyên. Điều này cũng đảm bảo khi làm trong modulo, nhân với nghịch đảo của 6 cho đúng giá trị.

Một cách nhìn trực giác khác: khoảng cách trung bình giữa hai hàng chọn ngẫu nhiên (khác nhau) là `(m+1)/3` — vì `Σ d(m−d) / C(m, 2) = (m+1)/3`.

In [4]:
from fractions import Fraction
for m in [2, 3, 10, 1000]:
    s = axis_sum_loop(m)
    assert s == axis_sum_closed(m) and (m - 1) * m * (m + 1) % 6 == 0
    print(f"m = {m}: tổng = {s}, trung bình = {Fraction(s, m * (m - 1) // 2)} (= (m+1)/3 = {Fraction(m + 1, 3)})")

m = 2: tổng = 1, trung bình = 1 (= (m+1)/3 = 1)
m = 3: tổng = 4, trung bình = 4/3 (= (m+1)/3 = 4/3)
m = 10: tổng = 165, trung bình = 11/3 (= (m+1)/3 = 11/3)
m = 1000: tổng = 166666500, trung bình = 1001/3 (= (m+1)/3 = 1001/3)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`2 × 2`, `k = 2`): mỗi cách xếp **là** một cặp ô; `C(2, 0) = 1`.

- Cặp cùng hàng / cùng cột (cách 1): 4 cặp; cặp chéo (cách 2): 2 cặp ⇒ `4·1 + 2·2 = 8` ✓.

Tách trục: phần hàng `= (Σ cặp hàng |Δr|) · n² = 1 · 4 = 4`; phần cột `= (Σ cặp cột |Δc|) · m² = 1 · 4 = 4` ⇒ `8` ✓. (Mỗi cặp hàng `(r1, r2)` kết hợp với **mọi** cột của mỗi ô: `n · n` cách.)

Ví dụ 2 (`1 × 4`, `k = 3`): phần cột `= Σ d(4 − d) = 3 + 4 + 3 = 10`, nhân `m² = 1`; `C(2, 1) = 2` ⇒ `20` ✓.

In [5]:
print('ví dụ 1:', (axis_sum_closed(2) * 4 + axis_sum_closed(2) * 4) * comb(2, 0))
print('ví dụ 2:', (axis_sum_closed(1) * 16 + axis_sum_closed(4) * 1) * comb(2, 1))

ví dụ 1: 8
ví dụ 2: 20


## 5. Cách 1 — Brute Force: liệt kê mọi cách xếp

In [6]:
import itertools

def distance_sum_brute(m, n, k):
    cells = [(r, c) for r in range(m) for c in range(n)]
    total = 0
    for S in itertools.combinations(cells, k):
        for p, q in itertools.combinations(S, 2):
            total += abs(p[0] - q[0]) + abs(p[1] - q[1])
    return total % MOD

for m, n, k, exp in examples:
    print(distance_sum_brute(m, n, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
20 | kỳ vọng 20


**Độ phức tạp:** `C(mn, k) · k²` — chỉ dùng cho lưới rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

`Đáp án = C(mn − 2, k − 2) · ( n² · Σ_{d=1}^{m−1} d(m−d) + m² · Σ_{d=1}^{n−1} d(n−d) )  (mod 10^9 + 7)`

Hai tổng trục có công thức đóng `m(m²−1)/6`, `n(n²−1)/6`; tính bằng số nguyên Python (chính xác) rồi mới lấy mod — hoặc dùng nghịch đảo của 6 nếu tính trong modulo.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt mọi cặp ô × `C(mn − 2, k − 2)`

In [7]:
def distance_sum_pairs(m, n, k):
    cells = [(r, c) for r in range(m) for c in range(n)]
    s = 0
    for i in range(len(cells)):
        r1, c1 = cells[i]
        for j in range(i + 1, len(cells)):
            r2, c2 = cells[j]
            s += abs(r1 - r2) + abs(c1 - c2)
    return s % MOD * comb_mod(m * n - 2, k - 2) % MOD

for m, n, k, exp in examples:
    print(distance_sum_pairs(m, n, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
20 | kỳ vọng 20


`O((mn)²)` — nhanh hơn liệt kê cách xếp rất nhiều, nhưng vẫn quá chậm với `mn = 10^5`.

### 7.2. Cách 3 — Tách trục, vòng lặp theo `d` + giai thừa modulo (lời giải chính)

In [8]:
def distance_sum_axis(m, n, k):
    N = m * n
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    def C(a, b):
        if b < 0 or b > a:
            return 0
        return fact[a] * pow(fact[b], MOD - 2, MOD) % MOD * pow(fact[a - b], MOD - 2, MOD) % MOD
    rows = sum(d * (m - d) for d in range(1, m)) % MOD
    cols = sum(d * (n - d) for d in range(1, n)) % MOD
    pair_sum = (rows * n % MOD * n + cols * m % MOD * m) % MOD
    return pair_sum * C(N - 2, k - 2) % MOD

for m, n, k, exp in examples:
    print(distance_sum_axis(m, n, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
20 | kỳ vọng 20


### 7.3. Cách 4 — Công thức đóng `O(1)` + `math.comb` (số nguyên lớn)

Python tính `comb(N − 2, k − 2)` chính xác bằng số nguyên lớn (vài chục nghìn chữ số khi `N = 10^5`) rồi mới lấy mod — vẫn đủ nhanh ở cỡ này.

In [9]:
def distance_sum_closed(m, n, k):
    pair_sum = n * n * (m * (m * m - 1) // 6) + m * m * (n * (n * n - 1) // 6)
    return pair_sum % MOD * (comb(m * n - 2, k - 2) % MOD) % MOD

for m, n, k, exp in examples:
    print(distance_sum_closed(m, n, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
20 | kỳ vọng 20


## 8. TRACE từng bước

Trace cách 3 với `m = 3, n = 2, k = 3`.

In [10]:
def trace(m, n, k):
    rows = [(d, m - d, d * (m - d)) for d in range(1, m)]
    cols = [(d, n - d, d * (n - d)) for d in range(1, n)]
    print('trục hàng (d, số cặp, d·số cặp):', rows, '-> tổng', sum(x[2] for x in rows))
    print('trục cột  (d, số cặp, d·số cặp):', cols, '-> tổng', sum(x[2] for x in cols))
    R, Cc = sum(x[2] for x in rows), sum(x[2] for x in cols)
    pair_sum = R * n * n + Cc * m * m
    print(f"tổng trên mọi cặp ô = {R}·{n}² + {Cc}·{m}² = {pair_sum}")
    c = comb(m * n - 2, k - 2)
    print(f"mỗi cặp ô xuất hiện trong C({m * n - 2}, {k - 2}) = {c} cách xếp -> đáp án {pair_sum * c % MOD}")
    print('vét cạn:', distance_sum_brute(m, n, k))

trace(3, 2, 3)

trục hàng (d, số cặp, d·số cặp): [(1, 2, 2), (2, 1, 2)] -> tổng 4
trục cột  (d, số cặp, d·số cặp): [(1, 1, 1)] -> tổng 1
tổng trên mọi cặp ô = 4·2² + 1·3² = 25
mỗi cặp ô xuất hiện trong C(4, 1) = 4 cách xếp -> đáp án 100
vét cạn: 100


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def distanceSum(self, m: int, n: int, k: int) -> int:
        MOD = 10 ** 9 + 7
        N = m * n
        fact = [1] * (N + 1)
        for i in range(1, N + 1):
            fact[i] = fact[i - 1] * i % MOD
        a, b = N - 2, k - 2
        comb_val = fact[a] * pow(fact[b] * fact[a - b] % MOD, MOD - 2, MOD) % MOD
        inv6 = pow(6, MOD - 2, MOD)
        rows = m * (m * m - 1) % MOD * inv6 % MOD          # Σ_{d} d(m-d)
        cols = n * (n * n - 1) % MOD * inv6 % MOD
        pair_sum = (rows * n % MOD * n + cols * m % MOD * m) % MOD
        return pair_sum * comb_val % MOD

for m, n, k, exp in examples:
    got = Solution().distanceSum(m, n, k)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
20 OK


## 10. Vì sao thuật toán đúng

1. **Đổi thứ tự tổng:** mỗi bộ (cách xếp `S`, cặp quân `{p, q}` trong `S`) tương ứng duy nhất với (cặp ô `{p, q}`, cách chọn `k − 2` ô còn lại) ⇒ hệ số `C(mn − 2, k − 2)` cho mọi cặp ô.
2. **Tách trục:** `|r1 − r2| + |c1 − c2|`; tổng phần hàng trên mọi cặp ô (không thứ tự) = `Σ_{cặp hàng không thứ tự} |Δr| · n²` (mỗi ô chọn cột tự do; cặp cùng hàng đóng góp 0). Tương tự phần cột.
3. **Công thức `m(m² − 1)/6`:** luôn là số nguyên (tích ba số liên tiếp `(m−1)m(m+1)` chia hết cho 6), nên dùng nghịch đảo của 6 trong modulo là hợp lệ.
4. **Tổ hợp modulo:** `p = 10^9 + 7` nguyên tố và `N < p` ⇒ các giai thừa khả nghịch.

## 11. Phân tích Time / Space Complexity

`N = mn ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê cách xếp | `C(N, k) · k²` | `O(k)` |
| 2. Mọi cặp ô | `O(N²)` | `O(N)` |
| 3. Tách trục + giai thừa | `O(N + m + n)` | `O(N)` |
| 4. Công thức đóng + `math.comb` | `O(1)` phép + `comb` số lớn | số lớn |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'pairs': distance_sum_pairs, 'axis': distance_sum_axis, 'closed': distance_sum_closed,
         'submit': lambda m, n, k: Solution().distanceSum(m, n, k)}
edge = [(1, 2, 2, 1), (2, 1, 2, 1), (2, 2, 4, 8), (3, 3, 2, 72), (1, 5, 5, 20)]
for m, n, k, exp in examples + edge:
    assert distance_sum_brute(m, n, k) == exp, (m, n, k, distance_sum_brute(m, n, k))
    for name, f in impls.items():
        assert f(m, n, k) == exp, (name, m, n, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu với vét cạn trên mọi lưới nhỏ

In [13]:
cnt = 0
for m in range(1, 4):
    for n in range(1, 4):
        for k in range(2, m * n + 1):
            exp = distance_sum_brute(m, n, k)
            for name, f in impls.items():
                assert f(m, n, k) == exp, (name, m, n, k)
            cnt += 1
import random
rng = random.Random(3426)
for _ in range(200):
    m, n = rng.randint(1, 25), rng.randint(1, 25)
    if m * n < 2:
        continue
    k = rng.randint(2, m * n)
    assert distance_sum_pairs(m, n, k) == distance_sum_axis(m, n, k) == distance_sum_closed(m, n, k) == Solution().distanceSum(m, n, k)
print('Khớp vét cạn trên', cnt, 'bộ (m, n ≤ 3); các cách khớp nhau trên 200 lưới tới 25 × 25.')

Khớp vét cạn trên 27 bộ (m, n ≤ 3); các cách khớp nhau trên 200 lưới tới 25 × 25.


### 12.3. Ca lớn `mn = 10^5`

In [14]:
import time
for m, n, k in [(1, 10 ** 5, 2), (10 ** 5, 1, 50000), (250, 400, 77777), (316, 316, 2)]:
    t = time.time(); a = Solution().distanceSum(m, n, k); t1 = time.time() - t
    t = time.time(); b = distance_sum_closed(m, n, k); t2 = time.time() - t
    assert a == b
    print(f"m={m:6d} n={n:6d} k={k:6d}: {a:10d} | giai thừa {t1:.2f}s | math.comb {t2:.2f}s")

m=     1 n=100000 k=     2:  665483338 | giai thừa 0.01s | math.comb 0.00s
m=100000 n=     1 k= 50000:  381759899 | giai thừa 0.01s | math.comb 0.08s
m=   250 n=   400 k= 77777:  644711958 | giai thừa 0.01s | math.comb 0.02s
m=   316 n=   316 k=     2:  291392010 | giai thừa 0.01s | math.comb 0.00s


Ghi chú hiệu năng: phần tốn nhất của lời giải nộp là dựng mảng giai thừa `N + 1` phần tử (`N ≤ 10^5`) — vài chục mili-giây. Ba lũy thừa nghịch đảo `pow(·, MOD − 2, MOD)` mỗi cái chỉ khoảng 30 phép nhân bình phương. Cách 4 dùng `math.comb` trên số nguyên lớn: với `k ≈ N/2`, `C(N − 2, k − 2)` có khoảng `N · log10(2) ≈ 30 000` chữ số, Python vẫn tính trong dưới 0,1 giây; nhưng ở ngôn ngữ không có số lớn, bắt buộc dùng giai thừa modulo như cách 3.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm cặp **có thứ tự** (mỗi cặp hai lần)

In [15]:
def wrong_ordered(m, n, k):
    cells = [(r, c) for r in range(m) for c in range(n)]
    s = sum(abs(p[0] - q[0]) + abs(p[1] - q[1]) for p in cells for q in cells)
    return s * comb(m * n - 2, k - 2) % MOD

print('sai :', wrong_ordered(2, 2, 2), '| đúng:', Solution().distanceSum(2, 2, 2))

sai : 16 | đúng: 8


**Vì sao sai:** mỗi cặp quân chỉ tính một lần.

### Lỗi 2 — Nhân với `C(mn, k)` (số cách xếp) thay vì `C(mn − 2, k − 2)`

In [16]:
def wrong_comb(m, n, k):
    pair_sum = n * n * (m * (m * m - 1) // 6) + m * m * (n * (n * n - 1) // 6)
    return pair_sum * comb(m * n, k) % MOD

print('sai :', wrong_comb(1, 4, 3), '| đúng:', Solution().distanceSum(1, 4, 3))

sai : 40 | đúng: 20


**Vì sao sai:** cặp ô `{p, q}` chỉ góp mặt trong những cách xếp **chứa cả hai** ô.

### Lỗi 3 — Quên nhân `n²` (`m²`) khi tách trục

In [17]:
def wrong_no_square(m, n, k):
    pair_sum = (m * (m * m - 1) // 6) + (n * (n * n - 1) // 6)
    return pair_sum * comb(m * n - 2, k - 2) % MOD

print('sai :', wrong_no_square(2, 2, 2), '| đúng:', Solution().distanceSum(2, 2, 2))

sai : 2 | đúng: 8


**Vì sao sai:** với mỗi cặp hàng `(r1, r2)`, hai ô có thể ở bất kỳ cột nào ⇒ `n · n` cặp ô.

### Lỗi 4 — Lấy mod rồi mới chia 6 bằng phép chia nguyên

In [18]:
def wrong_mod_div(m, n, k):
    rows = (m * (m * m - 1) % MOD) // 6            # SAI: chia sau khi đã mod
    cols = (n * (n * n - 1) % MOD) // 6
    pair_sum = (rows * n * n + cols * m * m) % MOD
    return pair_sum * comb_mod(m * n - 2, k - 2) % MOD

m, n, k = 1, 10 ** 5, 2
print('sai :', wrong_mod_div(m, n, k), '| đúng:', Solution().distanceSum(m, n, k))

sai : 165483334 | đúng: 665483338


**Vì sao sai:** `n(n² − 1) ≈ 10^{15}` vượt `MOD`; sau khi lấy mod, giá trị không còn chia hết cho 6. Phải chia **trước** (số nguyên Python) hoặc nhân **nghịch đảo** của 6 modulo `p`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng trên mọi cấu hình của tổng theo cặp" ⇒ đổi thứ tự: mỗi cặp nhân số cấu hình chứa nó.
- Khoảng cách Manhattan ⇒ tách trục, mỗi trục là bài 1 chiều.
- `Σ_{d} d(m − d) = m(m²−1)/6`.
- Chia trong modulo ⇒ nghịch đảo (số nguyên tố) — không dùng `//` sau khi đã `%`.

## 15. Biến thể của bài

1. **LC 1131/2615:** tổng khoảng cách giữa các điểm cho trước — sắp xếp + tiền tố theo từng trục.
2. Khoảng cách **Chebyshev** ⇒ không tách trục trực tiếp; xoay 45° biến Chebyshev thành Manhattan.
3. Quân **phân biệt** ⇒ nhân thêm `k!`.
4. Tổng **bình phương** khoảng cách Euclid ⇒ cũng tách trục: `Σ (Δr² + Δc²)`.

In [19]:
# Biến thể 4: tổng bình phương khoảng cách Euclid trên mọi cách xếp
def sq_dist_sum(m, n, k):
    rows = sum((m - d) * d * d for d in range(1, m))
    cols = sum((n - d) * d * d for d in range(1, n))
    return (rows * n * n + cols * m * m) * comb(m * n - 2, k - 2) % MOD

def sq_dist_brute(m, n, k):
    cells = [(r, c) for r in range(m) for c in range(n)]
    tot = 0
    for S in itertools.combinations(cells, k):
        for p, q in itertools.combinations(S, 2):
            tot += (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2
    return tot % MOD

for m in range(1, 4):
    for n in range(1, 4):
        for k in range(2, m * n + 1):
            assert sq_dist_sum(m, n, k) == sq_dist_brute(m, n, k)
print('Biến thể bình phương Euclid khớp vét cạn; 2x2, k=2:', sq_dist_sum(2, 2, 2))

Biến thể bình phương Euclid khớp vét cạn; 2x2, k=2: 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê cách xếp | mũ | Oracle |
| 2 | Mọi cặp ô | `O(N²)` | Trung gian |
| 3 | Tách trục + giai thừa mod | `O(N)` | **Nộp bài** |
| 4 | Công thức đóng + `math.comb` | `O(1)` + số lớn | Ngắn gọn trong Python |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `m = 1, n = 3, k = 2`.

**Bài 2 (tính tay):** `Σ_{d=1}^{4} d(5 − d)`.

**Bài 3 (điền TODO):** viết `sum_pairwise_manhattan(points)` cho tập điểm cho trước trong `O(P log P)`.

**Bài 4 (tự viết lại):** biến thể 3 — quân phân biệt (tổng trên mọi cách xếp có thứ tự).

**Bài 5 (mini challenge):** tổng khoảng cách Manhattan trên lưới **3 chiều** `a × b × c`.

In [20]:
def sum_pairwise_manhattan_todo(points):
    # TODO
    raise NotImplementedError

def distinct_pieces_todo(m, n, k):
    # TODO
    raise NotImplementedError

def distance_sum_3d_todo(a, b, c, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba ô trên một hàng. (2) Cặp: 1, 1, 2. (3) `C(1, 0) = 1` ⇒ 4.

**Bài 2.** (1) `4 + 6 + 6 + 4`. (2) Công thức `5 · 24 / 6`. (3) `20`.

**Bài 3.** (1) Tách trục. (2) Sắp tọa độ, điểm thứ `i` đóng góp `x_i · i − (tổng i điểm trước)`. (3) Cộng hai trục.

**Bài 4.** (1) Mỗi cách xếp quân giống nhau ứng với `k!` cách xếp quân phân biệt. (2) Nhân `k!`. (3) Kiểm tra bằng hoán vị nhỏ.

**Bài 5.** (1) Ba trục. (2) Trục `a` nhân `(bc)²`. (3) Hệ số `C(abc − 2, k − 2)`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
assert Solution().distanceSum(1, 3, 2) == 4 == distance_sum_brute(1, 3, 2)
assert sum(d * (5 - d) for d in range(1, 5)) == 20 == axis_sum_closed(5)
print('Bài 1: 4 | Bài 2: 20')

def sum_pairwise_manhattan(points):
    total = 0
    for axis in (0, 1):
        xs = sorted(p[axis] for p in points)
        pref = 0
        for i, x in enumerate(xs):
            total += x * i - pref
            pref += x
    return total

rng = random.Random(3)
for _ in range(300):
    pts = [(rng.randint(-5, 5), rng.randint(-5, 5)) for _ in range(rng.randint(1, 8))]
    assert sum_pairwise_manhattan(pts) == sum(abs(p[0] - q[0]) + abs(p[1] - q[1]) for p, q in itertools.combinations(pts, 2))
print('Bài 3 khớp vét cạn.')

Bài 1: 4 | Bài 2: 20
Bài 3 khớp vét cạn.


In [22]:
# Bài 4, 5
from math import factorial

def distinct_pieces(m, n, k):
    return Solution().distanceSum(m, n, k) * factorial(k) % MOD

def distinct_brute(m, n, k):
    cells = [(r, c) for r in range(m) for c in range(n)]
    tot = 0
    for S in itertools.permutations(cells, k):
        for p, q in itertools.combinations(S, 2):
            tot += abs(p[0] - q[0]) + abs(p[1] - q[1])
    return tot % MOD

def distance_sum_3d(a, b, c, k):
    ax = lambda L: L * (L * L - 1) // 6
    pair_sum = ax(a) * (b * c) ** 2 + ax(b) * (a * c) ** 2 + ax(c) * (a * b) ** 2
    return pair_sum * comb(a * b * c - 2, k - 2) % MOD

def brute_3d(a, b, c, k):
    cells = [(x, y, z) for x in range(a) for y in range(b) for z in range(c)]
    tot = 0
    for S in itertools.combinations(cells, k):
        for p, q in itertools.combinations(S, 2):
            tot += sum(abs(u - v) for u, v in zip(p, q))
    return tot % MOD

for (m, n, k) in [(2, 2, 2), (1, 3, 3), (2, 3, 3)]:
    assert distinct_pieces(m, n, k) == distinct_brute(m, n, k)
for (a, b, c, k) in [(2, 2, 2, 2), (1, 2, 3, 3), (2, 2, 2, 4)]:
    assert distance_sum_3d(a, b, c, k) == brute_3d(a, b, c, k)
assert distance_sum_3d(1, 2, 2, 2) == Solution().distanceSum(2, 2, 2)
print('Bài 4, 5 khớp vét cạn; 3D 2x2x2, k=2:', distance_sum_3d(2, 2, 2, 2))

Bài 4, 5 khớp vét cạn; 3D 2x2x2, k=2: 48


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pair_sum = n² · m(m²−1)/6 + m² · n(n²−1)/6          # tổng khoảng cách trên mọi cặp ô
ans = pair_sum · C(mn − 2, k − 2)  mod 1e9+7
C qua giai thừa + pow(x, MOD-2, MOD);  chia 6 qua inv6
```

**Checklist:**

- [ ] Cặp **không thứ tự**?
- [ ] Hệ số `C(mn − 2, k − 2)` (không phải `C(mn, k)`)?
- [ ] Nhân `n²` cho trục hàng, `m²` cho trục cột?
- [ ] Không dùng `//` sau khi đã lấy mod (dùng nghịch đảo)?
- [ ] `k = 2` ⇒ hệ số `C(mn − 2, 0) = 1`?